# Analisis Jangkauan Pos Pemadam Kebakaran Berbasis GIS

Notebook ini mendokumentasikan analisis untuk wilayah studi Surabaya Barat. Notebook memakai layer lokal yang sama dengan aplikasi Pygame dan memanggil fungsi analisis bersama dari `src/`, sehingga hasil cakupan tidak dihitung dengan implementasi terpisah.

Analisis membandingkan buffer **3.000 m** (baseline) dan **5.000 m** (skenario). Buffer dihitung sebagai jarak lurus dalam **EPSG:32749**, bukan jarak perjalanan. Routing OSRM, bila diaktifkan secara eksplisit, mengikuti jaringan jalan dan durasinya bukan informasi lalu lintas real-time.

## Data dan batas interpretasi

- Batas, jalan, dan entri pos berasal dari OpenStreetMap (OSM). Entri `amenity=fire_station` adalah data yang dipetakan di OSM, **bukan verifikasi oleh otoritas**. Atribusi: © OpenStreetMap contributors.
- Titik sampel adalah 13 titik sintetis yang dibuat pada jaringan jalan untuk demonstrasi. Titik tersebut bukan alamat atau representasi seluruh permukiman/populasi.
- Pos usulan adalah skenario hipotetis yang dibedakan dari entri OSM.
- Radius 3 km dan 5 km adalah parameter analisis proyek, bukan standar waktu layanan resmi.

Jalankan semua cell dari atas ke bawah. Notebook tidak mengunduh atau mengganti data secara diam-diam. Jika layer lokal belum tersedia, ikuti perintah persiapan data pada pesan error atau README.

## 1. Setup dan fungsi bersama

Di VS Code, pilih interpreter `.venv` proyek. Di terminal pada folder proyek, pasang dependensi dengan `python -m pip install -r requirements.txt`, lalu buka file notebook ini. Untuk Google Colab, clone atau unggah seluruh proyek beserta folder `data/`, lalu jalankan instalasi dependensi di root proyek. Cell berikut mencari root proyek agar impor `src` tetap konsisten.

In [ ]:
from pathlib import Path
import sys

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "config.py").is_file() and (path / "src").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Root proyek tidak ditemukan. Buka notebook dari proyek ini atau clone/unggah "
        "seluruh proyek ke Colab, lalu jadikan folder yang berisi config.py sebagai working directory."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from config import GEOGRAPHIC_CRS, MAP_FILE, PROJECTED_CRS, SAMPLES_FILE, STATIONS_FILE
from src.data_loader import DataUnavailableError, load_local_data
from src.routing_service import RoutingService
from src.spatial_analysis import calculate_coverage, station_buffer_geometries
from src.station_manager import StationManager

print(f"Root proyek: {project_root}")
print(f"CRS geografis: {GEOGRAPHIC_CRS}; CRS analisis: {PROJECTED_CRS}")

## 2. Memuat data lokal

Layer yang sudah disiapkan disimpan di `data/`. Membaca file lokal menghindari permintaan Overpass setiap kali notebook dibuka. Tidak ada fallback sintetis otomatis: jika data hilang atau rusak, hentikan analisis dan siapkan ulang layer secara eksplisit.

In [ ]:
try:
    boundary, roads, stations, samples = load_local_data(
        map_path=MAP_FILE,
        stations_path=STATIONS_FILE,
        samples_path=SAMPLES_FILE,
    )
except DataUnavailableError as error:
    raise DataUnavailableError(
        f"{error}\nPastikan file data proyek tersedia. Untuk menyiapkan ulang data, "
        "lihat instruksi `python -m src.data_loader --download` di README."
    ) from error

print(f"Batas studi: {len(boundary)} fitur")
print(f"Jaringan jalan: {len(roads)} fitur")
print(f"Entri pos OSM: {len(stations)} fitur")
print(f"Titik sampel sintetis: {len(samples)} fitur")

## 3. Pemeriksaan kualitas data

Periksa jumlah fitur, jenis geometri, CRS, bounds, geometri kosong/tidak valid, serta ID sampel. ID unik diperlukan agar satu titik yang masuk ke buffer beberapa pos tidak dihitung berulang.

In [ ]:
layers = {
    "boundary": boundary,
    "roads": roads,
    "stations": stations,
    "samples": samples,
}
quality_rows = []
for name, frame in layers.items():
    quality_rows.append({
        "layer": name,
        "features": len(frame),
        "geometry_types": ", ".join(sorted(frame.geometry.geom_type.dropna().unique())) or "none",
        "crs": str(frame.crs),
        "null_geometries": int(frame.geometry.isna().sum()),
        "empty_geometries": int(frame.geometry.is_empty.sum()),
        "invalid_geometries": int((~frame.geometry.is_valid).sum()),
        "bounds": frame.total_bounds.tolist() if not frame.empty else None,
    })
display(pd.DataFrame(quality_rows))

if samples.empty:
    raise ValueError("Analisis memerlukan setidaknya satu titik sampel.")
if "sample_id" not in samples or samples["sample_id"].isna().any():
    raise ValueError("Setiap titik sampel harus memiliki sample_id yang terisi.")
if samples["sample_id"].astype(str).duplicated().any():
    raise ValueError("sample_id harus unik agar jumlah cakupan dapat ditafsirkan dengan benar.")
if not stations.empty and stations["station_id"].astype(str).duplicated().any():
    raise ValueError("station_id pos harus unik.")

print("Sumber entri pos:")
if stations.empty:
    print("Tidak ada entri pos OSM pada layer lokal; ini tidak membuktikan tidak ada pos nyata.")
else:
    display(stations[["station_id", "name", "category", "source", "verification_status"]])

## 4. CRS dan transformasi koordinat

Data geografis disimpan pada EPSG:4326 (longitude/latitude). Sebelum menghitung buffer dalam meter, layer ditransformasikan dengan `to_crs()` ke EPSG:32749, zona UTM yang digunakan proyek untuk Surabaya. `to_crs()` mentransformasikan koordinat; CRS tidak sekadar ditempelkan pada geometri.

In [ ]:
if any(frame.crs is None for frame in layers.values()):
    raise ValueError("Semua layer harus mendeklarasikan CRS sebelum transformasi.")

boundary_m = boundary.to_crs(PROJECTED_CRS)
roads_m = roads.to_crs(PROJECTED_CRS)
stations_m = stations.to_crs(PROJECTED_CRS)
samples_m = samples.to_crs(PROJECTED_CRS)
print(f"CRS hasil transformasi: {samples_m.crs}")
print(f"Satuan koordinat proyeksi: meter; bounds sampel: {samples_m.total_bounds}")

## 5. Buffer dan cakupan baseline

Fungsi `calculate_coverage()` dari `src.spatial_analysis` mentransformasikan layer, membuat buffer pada jarak meter, lalu mencocokkan titik dengan buffer menggunakan spatial join `predicate="intersects"`. ID titik dihitung unik. Nilai ini mengukur kedekatan garis lurus, bukan akses jalan.

In [ ]:
coverage_before = calculate_coverage(samples, stations)
baseline_rows = [
    {
        "radius_km": radius / 1000,
        "covered_unique_samples": metric.covered_samples,
        "uncovered_samples": metric.uncovered_samples,
        "total_samples": metric.total_samples,
        "coverage_percent": metric.percentage,
    }
    for radius, metric in coverage_before.items()
]
baseline_table = pd.DataFrame(baseline_rows).sort_values("radius_km")
display(baseline_table)

buffer_summary = pd.DataFrame([
    {
        "radius_m": radius,
        "radius_km": radius / 1000,
        "buffer_features": len(station_buffer_geometries(stations, radius)),
        "projected_crs": str(PROJECTED_CRS),
    }
    for radius in (3_000, 5_000)
])
display(buffer_summary)
print("Buffer dibuat dalam EPSG:32749; satuan radius adalah meter.")

## 6. Peta cakupan

Peta berikut memperlihatkan layer batas, jalan, pos OSM, titik sampel, dan buffer. Titik hijau berada dalam setidaknya satu buffer pada radius yang sedang ditampilkan; titik jingga belum tercakup. Simbol hanya menggambarkan status data, bukan tingkat kepastian lokasi.

In [ ]:
def plot_coverage_map(station_layer, radius_m, metric, title):
    fig, ax = plt.subplots(figsize=(10, 8))
    boundary_m.plot(ax=ax, facecolor="#f7f7f2", edgecolor="#596b73", linewidth=1.2, zorder=1)
    roads_m.plot(ax=ax, color="#cbd4d7", linewidth=0.35, zorder=2)

    projected_stations = station_layer.to_crs(PROJECTED_CRS)
    buffer_shapes = station_buffer_geometries(station_layer, radius_m)
    if not buffer_shapes.empty:
        gpd.GeoDataFrame(geometry=buffer_shapes, crs=PROJECTED_CRS).plot(
            ax=ax,
            facecolor="#58a6b8" if radius_m == 3_000 else "#e6a84b",
            edgecolor="#267385" if radius_m == 3_000 else "#a16a1c",
            alpha=0.22,
            linewidth=0.8,
            zorder=3,
        )

    covered_ids = metric.covered_sample_ids
    covered_mask = samples_m["sample_id"].astype(str).isin(covered_ids)
    if (~covered_mask).any():
        samples_m.loc[~covered_mask].plot(ax=ax, color="#db8a36", markersize=22, zorder=5)
    if covered_mask.any():
        samples_m.loc[covered_mask].plot(ax=ax, color="#238b68", markersize=22, zorder=6)

    if not projected_stations.empty:
        real_mask = projected_stations.get(
            "category", pd.Series("REAL-OSM", index=projected_stations.index)
        ).astype(str).ne("PROPOSED")
        if real_mask.any():
            projected_stations.loc[real_mask].plot(
                ax=ax, color="#bd3d3d", marker="^", markersize=65, zorder=7
            )
        if (~real_mask).any():
            projected_stations.loc[~real_mask].plot(
                ax=ax, color="#7545ad", marker="s", markersize=65, zorder=8
            )

    ax.legend(handles=[
        Patch(
            facecolor="#58a6b8" if radius_m == 3_000 else "#e6a84b",
            alpha=0.35,
            label=f"Buffer {radius_m / 1000:g} km",
        ),
        Line2D([], [], marker="^", linestyle="", color="#bd3d3d", label="Entri pos OSM"),
        Line2D([], [], marker="s", linestyle="", color="#7545ad", label="Pos usulan"),
        Line2D([], [], marker="o", linestyle="", color="#238b68", label="Sampel tercakup"),
        Line2D([], [], marker="o", linestyle="", color="#db8a36", label="Sampel belum tercakup"),
        Line2D([], [], color="#cbd4d7", label="Jalan OSM"),
    ], loc="best")
    ax.set_title(title)
    ax.set_axis_off()
    ax.set_aspect("equal")
    plt.tight_layout()
    return fig, ax

for radius in (3_000, 5_000):
    plot_coverage_map(
        stations,
        radius,
        coverage_before[radius],
        f"Cakupan baseline: buffer {radius / 1000:g} km",
    )
    plt.show()

## 7. Skenario pos usulan dan perbandingan

Heuristik yang dapat dijelaskan: dari titik sampel yang belum tercakup buffer 3 km pada baseline, pilih titik dengan jarak lurus terjauh terhadap pos OSM terdekat. Titik sampel tersebut dipakai sebagai **kandidat simulasi** karena lokasinya berada di jalan pada data demonstrasi. Ini bukan rekomendasi pembangunan; kelayakan lahan, akses, risiko, kebutuhan operasional, dan verifikasi lapangan tidak dianalisis.

Jika tidak ada titik sampel yang belum tercakup atau tidak ada entri pos OSM, notebook tidak mengarang kandidat.

In [ ]:
baseline_3km = coverage_before[3_000]
uncovered_3km = samples.loc[
    ~samples["sample_id"].astype(str).isin(baseline_3km.covered_sample_ids)
].copy()
station_manager = StationManager(stations)
candidate_point = None

if stations.empty:
    print("Tidak ada pos OSM untuk menghitung jarak ke pos terdekat; kandidat tidak dibuat.")
elif uncovered_3km.empty:
    print("Semua titik sampel sudah tercakup pada baseline 3 km; kandidat tidak dibuat.")
else:
    uncovered_m = uncovered_3km.to_crs(PROJECTED_CRS)
    station_geometries_m = stations_m.geometry
    nearest_station_distance_m = uncovered_m.geometry.apply(
        lambda point: station_geometries_m.distance(point).min()
    )
    candidate_index = nearest_station_distance_m.idxmax()
    candidate_point = uncovered_3km.loc[candidate_index].geometry
    proposal = station_manager.add_proposed(candidate_point)
    print(f"Kandidat simulasi: {proposal.name} pada titik {uncovered_3km.loc[candidate_index]['sample_id']}")
    print("Status: HIPOTETIS; bukan fasilitas aktual atau rekomendasi pembangunan.")

scenario_stations = station_manager.all_stations()
coverage_after = calculate_coverage(samples, scenario_stations)
comparison_rows = []
for radius in (3_000, 5_000):
    for scenario_name, metrics in (("Sebelum", coverage_before), ("Sesudah", coverage_after)):
        metric = metrics[radius]
        comparison_rows.append({
            "skenario": scenario_name,
            "radius_km": radius / 1000,
            "pos_osm": len(stations),
            "pos_usulan": station_manager.proposed_count if scenario_name == "Sesudah" else 0,
            "titik_tercakup_unik": metric.covered_samples,
            "titik_belum_tercakup": metric.uncovered_samples,
            "persentase": metric.percentage,
        })
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)

In [ ]:
plot_coverage_map(
    scenario_stations,
    3_000,
    coverage_after[3_000],
    "Skenario sesudah penambahan pos usulan (jika kandidat tersedia)",
)
plt.show()

## 8. Routing OSRM (opsional)

Agar notebook yang dijalankan berurutan tidak membuat permintaan jaringan tanpa sengaja, contoh routing dinonaktifkan secara default. Ubah `RUN_OSRM_EXAMPLE` menjadi `True` untuk satu perbandingan tujuan menggunakan layanan publik OSRM. Routing mencoba membandingkan entri pos yang tersedia; kegagalan dan rute yang tidak dapat di-snap tetap dilaporkan. Estimasi durasi bukan lalu lintas real-time atau jaminan respons darurat.

In [ ]:
RUN_OSRM_EXAMPLE = False
ROUTE_SAMPLE_ID = str(samples.iloc[0]["sample_id"])
route_comparison = None

if not RUN_OSRM_EXAMPLE:
    print("Routing dilewati. Ubah RUN_OSRM_EXAMPLE=True jika ingin mengirim permintaan OSRM.")
elif stations.empty:
    print("Routing dilewati karena tidak ada entri pos OSM pada data lokal.")
else:
    destination_rows = samples.loc[samples["sample_id"].astype(str) == ROUTE_SAMPLE_ID]
    if destination_rows.empty:
        raise ValueError(f"ROUTE_SAMPLE_ID tidak ditemukan: {ROUTE_SAMPLE_ID}")
    route_comparison = RoutingService().compare_routes(
        stations,
        destination_rows.geometry.iloc[0],
    )
    print(route_comparison.message)
    route_table = pd.DataFrame([
        {
            "station_id": route.station_id,
            "station_name": route.station_name,
            "success": route.success,
            "distance_km": route.distance_meters / 1000 if route.distance_meters is not None else None,
            "estimated_minutes": route.duration_seconds / 60 if route.duration_seconds is not None else None,
            "reason": route.reason,
        }
        for route in route_comparison.routes
    ])
    display(route_table)
    if route_comparison.fastest_station_id is not None:
        print(f"ID pos dengan durasi terendah di antara rute berhasil: {route_comparison.fastest_station_id}")
    if route_comparison.geometry is not None:
        route_m = gpd.GeoSeries([route_comparison.geometry], crs=GEOGRAPHIC_CRS).to_crs(PROJECTED_CRS)
        fig, ax = plot_coverage_map(
            stations,
            3_000,
            coverage_before[3_000],
            "Rute OSRM yang dipilih (jika tersedia)",
        )
        route_m.plot(ax=ax, color="#d64b36", linewidth=2.2, zorder=9)
        plt.show()

## 9. Interpretasi dan keterbatasan

Gunakan tabel yang dihasilkan cell di atas untuk menulis interpretasi: bandingkan cakupan 3 km dan 5 km; lalu bandingkan baseline dengan skenario pos usulan. Jumlah dan persentase dihitung dari **titik sampel sintetis**, bukan jumlah rumah tangga atau penduduk. Nilai hasil harus berasal dari eksekusi data lokal, bukan disalin sebagai angka umum.

Keterbatasan utama: kelengkapan dan posisi entri OSM belum diverifikasi otoritas; sampel bukan data permukiman aktual; buffer adalah jarak lurus; dan OSRM publik dapat gagal, memiliki batas layanan, serta tidak memberi data lalu lintas real-time. Kandidat usulan hanya heuristik untuk demonstrasi dan memerlukan penilaian spasial, operasional, dan perencanaan sebelum dapat dipertimbangkan sebagai lokasi nyata.

Peta dan data OSM: © OpenStreetMap contributors, tersedia di bawah Open Database License (ODbL).